In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
#Mô phỏng cơ chế mặt nạ trong Decoder và dự đoán xác suất từ tiếp theo
#Cơ chế Masking (Mặt nạ chặn tương lai)
#Giúp đảm bảo tính nhân quả, ngăn mô hình nhìn trộm các token tương lai
def create_look_ahead_mask(size):
    # Tạo ma trận tam giác: nửa trên bằng 1 (vị trí bị chặn), nửa dưới và đường chéo bằng 0
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask

#Lớp Decoder Block
class DecoderBlock(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        self.masked_attention = nn.Linear(d_model, d_model)
        self.cross_attention = nn.Linear(d_model, d_model) # Kết nối với Encoder
        self.ffn = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, encoder_output, mask):
        #Masked Self-Attention (Chỉ nhìn về quá khứ)
        #Mask này được áp dụng trước Softmax
        attn_out = self.masked_attention(x)
        x = self.norm(x + attn_out)
        
        #Cross Attention (Lấy thông tin từ Encoder)
        #Query từ Decoder, Key & Value từ Encoder
        cross_out = self.cross_attention(encoder_output)
        x = self.norm(x + cross_out)
        
        #Feed Forward
        return self.norm(x + self.ffn(x))

#Prediction Head (Đầu dự đoán xác suất)
#Chuyển vector 512 chiều thành xác suất của hàng vạn từ
class PredictionHead(nn.Module):
    def __init__(self, d_model=512, vocab_size=37000):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # Trả về điểm số (Logits) cho mỗi từ trong bộ từ điển
        return self.linear(x)

In [3]:
# - Chuỗi đích hiện tại có 9 từ (đang dịch dở)
# - Encoder gửi sang thông tin của 10 từ nguồn
seq_len_target = 9
seq_len_source = 10
d_model = 512
vocab_size = 37000

#Tạo Input
target_input = torch.randn(1, seq_len_target, d_model)
encoder_memory = torch.randn(1, seq_len_source, d_model) # Thông tin từ Encoder

#Tạo Mask
mask = create_look_ahead_mask(seq_len_target)
print(f"LOOK-AHEAD MASK  : Ma trận mặt nạ (True là vị trí AI không được nhìn):\n{mask}\n")

#Chạy qua Decoder Block
decoder = DecoderBlock(d_model)
encoder_for_decoder = encoder_memory[:, :seq_len_target, :]
decoder_output = decoder(
    target_input,
    encoder_for_decoder,
    mask
)
print(f"DECODER OUTPUT : Kích thước vector sau Decoder: {decoder_output.shape} (Vẫn giữ 512 chiều)\n")

# Dự đoán từ tiếp theo qua Prediction Head
predict_head = PredictionHead(d_model, vocab_size)
logits = predict_head(decoder_output) # (1, 9, 37000)

# Lấy dự đoán cho token cuối cùng trong chuỗi
next_token_logits = logits[:, -1, :] 
print("PREDICTION (Xác suất từ kế tiếp) : ")
print(f"Kích thước vector xác suất: {next_token_logits.shape} (Ứng với {vocab_size} từ trong từ điển)")
print(f"Token được chọn (ID): {torch.argmax(next_token_logits).item()}")

LOOK-AHEAD MASK  : Ma trận mặt nạ (True là vị trí AI không được nhìn):
tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True,  True],
        [False, False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False, False]])

DECODER OUTPUT : Kích thước vector sau Decoder: torch.Size([1, 9, 512]) (Vẫn giữ 512 chiều)

PREDICTION (Xác suất từ kế tiếp) : 
Kích thước vector xác suất: torch.Size([1, 37000]) (Ứng với 37000 từ trong từ điển)
Token được chọn (ID): 8346
